In [13]:
import trimesh
import numpy as np
from read_mesh import facenormal_angle,powerratio_iam
from pv_index import pv_perf_index
from sun import get_sun_direction
from datetime import datetime
import pytz

sun_dir=np.array([-7,1,2])

mesh =trimesh.load_mesh('data/stl_models/car.stl')



Calculate Index

In [14]:
index, best_mesh= pv_perf_index(mesh,sun_dir,max_area=6.0) 

print("Performance index:",index)
print("used area:", round((sum(best_mesh.facets_area)/10e5),3),"m2")


Performance index: 0.8756110333903023
used area: 2.556 m2


Visualize Selected Area

In [15]:
#color the selected mesh in red
best_mesh.visual.face_colors = [255, 0, 0, 150]

# the original mesh in green
mesh.visual.face_colors = [0, 255, 0, 150] 

# create a scene with both meshes
scene = trimesh.Scene([mesh, best_mesh])



scene.show(notebook=True)

In [16]:
from helpers.visualize_weight import color_faces_by_weight
from read_mesh import shadowing_factor

top_mesh = mesh#sky_face(mesh,sun_dir)
angles = facenormal_angle(top_mesh,sun_dir)

shadow_mask = shadowing_factor(top_mesh, sun_dir*-1)

power_weight = powerratio_iam(angles)
power_weight[shadow_mask]=0

color_faces_by_weight(top_mesh, power_weight, gamma=1.0).show()


In [17]:
color_faces_by_weight(top_mesh, power_weight, gamma=1.0).show()

visualize shadowed areas

In [18]:
shadowed_faces_indices = np.where(shadow_mask)[0]
illuminated_faces_indices = np.where(~shadow_mask)[0]

print(f"Gesamtanzahl Facetten: {len(mesh.faces)}")
print(f"Beschattete Facetten: {len(shadowed_faces_indices)}")
print(f"Unbeschattete Facetten: {len(illuminated_faces_indices)}")

# Optional: Visualisierung
# Wir können die beschatteten Facetten rot einfärben.
face_colors = np.array([50, 50, 50, 255] * len(mesh.faces), dtype=np.uint8).reshape((-1, 4))
face_colors[shadowed_faces_indices] = [255, 0, 0, 255] # Rot für beschattet
face_colors[illuminated_faces_indices] = [0, 255, 0, 255] # Grün für unbeschattet

mesh.visual.face_colors = face_colors
mesh.show() 

Gesamtanzahl Facetten: 11784
Beschattete Facetten: 5993
Unbeschattete Facetten: 5791
